# 10. Selected Feature Stability Check
## Final Test 직전 약식 점검

이번 단계는 **새 Feature 탐색 / Embedding / HPO를 하지 않습니다.**

목적은 하나입니다.

> 09-02에서 확정한 Hard Two-stage가 주요 feature group을 빼도 흔들리지 않는지 확인.

비교:

```text
FULL
NO_MARKET
NO_TRANSFER_CONTEXT
NO_MODEL_A_CONTEXT
```

판단 규칙도 미리 고정합니다.

- 어떤 ablation이 FULL보다 평균 MAE를 **0.03 이상 개선**
- 그리고 **4개 Fold 중 3개 이상**에서 개선

할 때만 구조 단순화를 검토합니다.

그 외에는 **FULL 유지 → 즉시 11 Final Test**로 갑니다.

Final Test(`2024-25 → 2025-26`)는 이 Notebook에서 절대 읽지 않습니다.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 0. Google Drive

Drive가 아직 안 붙어 있으면 먼저 실행:

```python
from google.colab import drive
drive.mount('/content/drive')
```

In [4]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive"
)

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(
        "Google Drive가 마운트되어 있지 않습니다. "
        "위의 drive.mount('/content/drive')를 먼저 실행하세요."
    )

print(
    "Google Drive mounted:",
    DRIVE_ROOT.exists(),
)

Google Drive mounted: True


In [5]:
import json
import random
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_fscore_support,
    r2_score,
    roc_auc_score,
)

warnings.filterwarnings(
    "ignore"
)

SEED = 42
MATERIAL_MAE_DELTA = 0.03

random.seed(SEED)
np.random.seed(SEED)

pd.set_option(
    "display.max_columns",
    120,
)

In [6]:
try:
    import catboost

    from catboost import (
        CatBoostClassifier,
        CatBoostRegressor,
    )

except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "catboost",
        ]
    )

    import catboost

    from catboost import (
        CatBoostClassifier,
        CatBoostRegressor,
    )


try:
    import torch
    USE_GPU = bool(
        torch.cuda.is_available()
    )

except ImportError:
    USE_GPU = False


DEVICE_PARAMS = (
    {
        "task_type": "GPU",
        "devices": "0",
    }
    if USE_GPU
    else {
        "task_type": "CPU",
    }
)

print(
    "CatBoost:",
    catboost.__version__,
)

print(
    "GPU:",
    USE_GPU,
)

CatBoost: 1.2.10
GPU: True


# Part A. Artifact 자동 탐색

In [7]:
REQUIRED_SNAPSHOT = (
    "09_01A_snapshot_conservative_labels.csv"
)


def find_artifact_dir():
    candidates = []

    for project_dir in (
        DRIVE_ROOT.glob(
            "next_season_goal_prediction*"
        )
    ):
        artifact_dir = (
            project_dir
            / "artifacts"
        )

        snapshot = (
            artifact_dir
            / REQUIRED_SNAPSHOT
        )

        if snapshot.exists():
            candidates.append(
                artifact_dir
            )

    if not candidates:
        raise FileNotFoundError(
            f"{REQUIRED_SNAPSHOT}를 찾지 못했습니다."
        )

    candidates.sort(
        key=lambda p:
            (
                p
                / REQUIRED_SNAPSHOT
            ).stat().st_mtime,
        reverse=True,
    )

    return candidates[0]


ARTIFACT_DIR = (
    find_artifact_dir()
)

SNAPSHOT_PATH = (
    ARTIFACT_DIR
    / REQUIRED_SNAPSHOT
)

print(
    "Artifact dir:",
    ARTIFACT_DIR,
)

print(
    "Snapshot:",
    SNAPSHOT_PATH,
)

Artifact dir: /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts
Snapshot: /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01A_snapshot_conservative_labels.csv


In [8]:
df = pd.read_csv(
    SNAPSHOT_PATH,
    low_memory=False,
)

print(
    "Shape:",
    df.shape,
)

print(
    "Audited corrections:",
    int(
        df[
            "label_audit_changed"
        ].sum()
    ),
)

Shape: (23353, 124)
Audited corrections: 20


# Part B. Final Test Lock + Historical Features

In [9]:
LOCKED_TEST_INPUT_SEASON = (
    "2024-2025"
)

LOCKED_TEST_TARGET_SEASON = (
    "2025-2026"
)

assert (
    LOCKED_TEST_INPUT_SEASON
    not in set(
        df[
            "season"
        ].astype(str)
    )
)

print(
    "✅ Final Test locked:",
    LOCKED_TEST_INPUT_SEASON,
    "→",
    LOCKED_TEST_TARGET_SEASON,
)

✅ Final Test locked: 2024-2025 → 2025-2026


In [10]:
df[
    "season_start"
] = (
    df[
        "season"
    ]
    .astype(str)
    .str[:4]
    .astype(int)
)

df[
    "target_year"
] = (
    df[
        "target_season"
    ]
    .astype(str)
    .str[:4]
    .astype(int)
)

df = (
    df
    .sort_values(
        [
            "player",
            "season_start",
        ]
    )
    .reset_index(
        drop=True
    )
)

g = df.groupby(
    "player",
    sort=False,
)

df[
    "goals_3yr_mean"
] = g[
    "goals"
].transform(
    lambda s:
        s.shift(1)
        .rolling(
            3,
            min_periods=1,
        )
        .mean()
)

df[
    "goals_per90_3yr_mean"
] = g[
    "goals_per90"
].transform(
    lambda s:
        s.shift(1)
        .rolling(
            3,
            min_periods=1,
        )
        .mean()
)

df[
    "goals_3yr_max"
] = g[
    "goals"
].transform(
    lambda s:
        s.shift(1)
        .rolling(
            3,
            min_periods=1,
        )
        .max()
)

HISTORICAL = [
    "goals_3yr_mean",
    "goals_per90_3yr_mean",
    "goals_3yr_max",
]

df[
    HISTORICAL
] = (
    df[
        HISTORICAL
    ]
    .fillna(0.0)
)

# Part C. Feature Groups

In [11]:
BASE = [
    "age",
    "starts",
    "minutes",
    "goals",
    "assists",
    "non_penalty_goals",
    "penalty_goals",
    "penalty_attempts",
    "goals_per90",
    "assists_per90",
    "goal_contrib_per90",
]

TEAM = [
    "old_team_rank_pct",
    "old_team_points_per_game",
    "old_team_goal_diff_per_game",
]

MARKET = [
    "market_value_known",
    "log_market_value",
    "market_value_percentile",
    "market_value_vs_position_median",
    "market_value_growth_6m",
    "market_value_growth_12m",
    "market_value_vs_peak",
]

TRANSFER = [
    "changed_team_preseason",
    "same_league_transfer",
    "league_changed",
    "country_changed",
    "is_loan_preseason",
    "days_since_transfer",
]

NEW_TEAM = [
    "new_team_prev_rank_pct",
    "new_team_prev_points_per_game",
    "new_team_prev_goal_diff_per_game",
    "team_rank_change",
    "team_points_change",
    "team_goal_diff_change",
    "new_team_strength_missing",
]

MODEL_A_CONTEXT = [
    "transfer_event_preseason",
    "destination_in_big5",
]

CATEGORICAL = [
    "league",
    "position_group",
]

CORE = (
    BASE
    + HISTORICAL
    + TEAM
)

FULL_B = (
    CORE
    + MARKET
    + TRANSFER
    + NEW_TEAM
)

FULL_A = (
    FULL_B
    + MODEL_A_CONTEXT
)


FEATURE_VARIANTS = {
    "FULL": {
        "a_numeric": FULL_A,
        "b_numeric": FULL_B,
    },

    "NO_MARKET": {
        "a_numeric": (
            CORE
            + TRANSFER
            + NEW_TEAM
            + MODEL_A_CONTEXT
        ),
        "b_numeric": (
            CORE
            + TRANSFER
            + NEW_TEAM
        ),
    },

    "NO_TRANSFER_CONTEXT": {
        "a_numeric": (
            CORE
            + MARKET
        ),
        "b_numeric": (
            CORE
            + MARKET
        ),
    },

    "NO_MODEL_A_CONTEXT": {
        "a_numeric": FULL_B,
        "b_numeric": FULL_B,
    },
}


for name, cfg in (
    FEATURE_VARIANTS.items()
):
    print(
        f"{name:<22}",
        "A:",
        len(
            cfg[
                "a_numeric"
            ]
        ),
        "B:",
        len(
            cfg[
                "b_numeric"
            ]
        ),
    )

FULL                   A: 39 B: 37
NO_MARKET              A: 32 B: 30
NO_TRANSFER_CONTEXT    A: 24 B: 24
NO_MODEL_A_CONTEXT     A: 37 B: 37


## Leakage Guard

In [12]:
FORBIDDEN = {
    "matched_next",
    "matched_next_original",
    "matched_next_audited",
    "next_goals",
    "next_goals_original",
    "next_goals_audited",
    "next_10plus",
    "next_10plus_original",
    "next_10plus_audited",
    "label_audit_changed",
}

for cfg in (
    FEATURE_VARIANTS.values()
):
    assert not (
        FORBIDDEN
        & set(
            cfg[
                "a_numeric"
            ]
        )
    )

    assert not (
        FORBIDDEN
        & set(
            cfg[
                "b_numeric"
            ]
        )
    )

print(
    "✅ Leakage guard passed."
)

✅ Leakage guard passed.


# Part D. Development Data / Folds

In [13]:
dev = (
    df[
        df[
            "target_year"
        ].ge(2017)
    ]
    .copy()
)

dev[
    "target_presence"
] = (
    dev[
        "matched_next_audited"
    ]
    .astype(int)
)

dev[
    "target_goals"
] = (
    dev[
        "next_goals_audited"
    ]
    .astype(float)
)


OUTER_VAL_SEASONS = [
    "2020-2021",
    "2021-2022",
    "2022-2023",
    "2023-2024",
]


def split_outer(
    data,
    val_season,
):
    val_start = int(
        val_season[:4]
    )

    return (
        data[
            data[
                "season_start"
            ]
            < val_start
        ].copy(),

        data[
            data[
                "season_start"
            ]
            == val_start
        ].copy(),
    )


def split_calibration(
    outer_train,
):
    last_train_season = (
        outer_train[
            "season_start"
        ].max()
    )

    cal_train = (
        outer_train[
            outer_train[
                "season_start"
            ]
            < last_train_season
        ]
        .copy()
    )

    cal_val = (
        outer_train[
            outer_train[
                "season_start"
            ]
            == last_train_season
        ]
        .copy()
    )

    return (
        cal_train,
        cal_val,
    )


print(
    "Development rows:",
    len(
        dev
    ),
)

Development rows: 7572


# Part E. Shared Helpers

In [14]:
def prepare_X(
    data,
    numeric_features,
):
    X = (
        data[
            numeric_features
            + CATEGORICAL
        ]
        .copy()
    )

    for col in (
        CATEGORICAL
    ):
        X[col] = (
            X[col]
            .fillna(
                "__MISSING__"
            )
            .astype(str)
        )

    for col in (
        numeric_features
    ):
        X[col] = pd.to_numeric(
            X[col],
            errors="coerce",
        )

    cat_indices = [
        X.columns.get_loc(
            col
        )
        for col in (
            CATEGORICAL
        )
    ]

    return (
        X,
        cat_indices,
    )

In [15]:
THRESHOLD_GRID = np.round(
    np.arange(
        0.20,
        0.951,
        0.01,
    ),
    2,
)


def threshold_stats(
    y,
    prob,
    threshold,
):
    pred = (
        np.asarray(
            prob
        )
        >= threshold
    ).astype(int)

    (
        precision,
        recall,
        f1,
        support,
    ) = precision_recall_fscore_support(
        y,
        pred,
        labels=[
            0,
            1,
        ],
        zero_division=0,
    )

    return {
        "macro_f1": (
            f1_score(
                y,
                pred,
                average="macro",
                zero_division=0,
            )
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                y,
                pred,
            )
        ),
        "exit_recall": (
            recall[0]
        ),
        "presence_recall": (
            recall[1]
        ),
    }


def choose_threshold(
    y,
    prob,
):
    rows = []

    for th in (
        THRESHOLD_GRID
    ):
        m = threshold_stats(
            y,
            prob,
            th,
        )

        rows.append({
            "threshold": th,
            **m,
        })

    table = pd.DataFrame(
        rows
    )

    best = (
        table
        .sort_values(
            [
                "macro_f1",
                "balanced_accuracy",
                "presence_recall",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .iloc[0]
    )

    return float(
        best[
            "threshold"
        ]
    )

# Part F. Model A / Model B

In [16]:
A_MAX_ITER = 2000
A_PATIENCE = 60

B_MAX_ITER = 2000
B_PATIENCE = 50

A_PARAMS = {
    "depth": 6,
    "learning_rate": 0.03,
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
}

B_PARAMS = {
    "depth": 6,
    "learning_rate": 0.03,
    "loss_function": "RMSE",
}

In [17]:
def fit_model_a(
    outer_train,
    outer_val,
    numeric_features,
    seed,
):
    (
        cal_train,
        cal_val,
    ) = split_calibration(
        outer_train
    )

    (
        X_ct,
        cat_idx,
    ) = prepare_X(
        cal_train,
        numeric_features,
    )

    (
        X_cv,
        _,
    ) = prepare_X(
        cal_val,
        numeric_features,
    )

    selector = CatBoostClassifier(
        iterations=A_MAX_ITER,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        **DEVICE_PARAMS,
        **A_PARAMS,
    )

    selector.fit(
        X_ct,
        cal_train[
            "target_presence"
        ],
        cat_features=(
            cat_idx
        ),
        eval_set=(
            X_cv,
            cal_val[
                "target_presence"
            ],
        ),
        early_stopping_rounds=(
            A_PATIENCE
        ),
        use_best_model=True,
        verbose=False,
    )

    iterations = max(
        1,
        int(
            selector.get_best_iteration()
        )
        + 1,
    )

    cal_prob = (
        selector.predict_proba(
            X_cv
        )[:, 1]
    )

    threshold = (
        choose_threshold(
            cal_val[
                "target_presence"
            ].to_numpy(),
            cal_prob,
        )
    )

    (
        X_train,
        cat_idx,
    ) = prepare_X(
        outer_train,
        numeric_features,
    )

    (
        X_val,
        _,
    ) = prepare_X(
        outer_val,
        numeric_features,
    )

    model = CatBoostClassifier(
        iterations=iterations,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        **DEVICE_PARAMS,
        **A_PARAMS,
    )

    model.fit(
        X_train,
        outer_train[
            "target_presence"
        ],
        cat_features=(
            cat_idx
        ),
        verbose=False,
    )

    prob = (
        model.predict_proba(
            X_val
        )[:, 1]
    )

    return (
        prob,
        threshold,
    )

In [18]:
def fit_model_b(
    outer_train_all,
    outer_val_all,
    numeric_features,
    seed,
):
    positive_train = (
        outer_train_all[
            outer_train_all[
                "target_presence"
            ].eq(1)
        ]
        .copy()
    )

    (
        cal_train,
        cal_val,
    ) = split_calibration(
        positive_train
    )

    (
        X_ct,
        cat_idx,
    ) = prepare_X(
        cal_train,
        numeric_features,
    )

    (
        X_cv,
        _,
    ) = prepare_X(
        cal_val,
        numeric_features,
    )

    selector = CatBoostRegressor(
        iterations=B_MAX_ITER,
        eval_metric="MAE",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        **DEVICE_PARAMS,
        **B_PARAMS,
    )

    selector.fit(
        X_ct,
        cal_train[
            "target_goals"
        ],
        cat_features=(
            cat_idx
        ),
        eval_set=(
            X_cv,
            cal_val[
                "target_goals"
            ],
        ),
        early_stopping_rounds=(
            B_PATIENCE
        ),
        use_best_model=True,
        verbose=False,
    )

    iterations = max(
        1,
        int(
            selector.get_best_iteration()
        )
        + 1,
    )

    (
        X_train,
        cat_idx,
    ) = prepare_X(
        positive_train,
        numeric_features,
    )

    (
        X_val,
        _,
    ) = prepare_X(
        outer_val_all,
        numeric_features,
    )

    model = CatBoostRegressor(
        iterations=iterations,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        **DEVICE_PARAMS,
        **B_PARAMS,
    )

    model.fit(
        X_train,
        positive_train[
            "target_goals"
        ],
        cat_features=(
            cat_idx
        ),
        verbose=False,
    )

    pred = np.clip(
        model.predict(
            X_val
        ),
        0,
        None,
    )

    return pred

# Part G. Hard Two-stage 평가

In [19]:
def safe_mae(
    y,
    pred,
    mask,
):
    if not np.any(
        mask
    ):
        return np.nan

    return mean_absolute_error(
        y[
            mask
        ],
        pred[
            mask
        ],
    )


def evaluate(
    val_df,
    prob,
    threshold,
    conditional_pred,
):
    y = (
        val_df[
            "target_goals"
        ]
        .to_numpy(
            dtype=float
        )
    )

    presence = (
        val_df[
            "target_presence"
        ]
        .to_numpy(
            dtype=int
        )
    )

    hard_pred = np.where(
        prob
        >= threshold,
        conditional_pred,
        0.0,
    )

    changed = (
        val_df[
            "changed_team_preseason"
        ]
        .fillna(0)
        .eq(1)
        .to_numpy()
    )

    exit_mask = (
        presence
        == 0
    )

    presence_mask = (
        presence
        == 1
    )

    ten_mask = (
        y
        >= 10
    )

    twenty_mask = (
        y
        >= 20
    )

    class_m = threshold_stats(
        presence,
        prob,
        threshold,
    )

    return {
        "mae": (
            mean_absolute_error(
                y,
                hard_pred,
            )
        ),
        "rmse": (
            mean_squared_error(
                y,
                hard_pred,
            )
            ** 0.5
        ),
        "r2": (
            r2_score(
                y,
                hard_pred,
            )
        ),
        "bias": float(
            np.mean(
                hard_pred
                - y
            )
        ),
        "exit_mae": (
            safe_mae(
                y,
                hard_pred,
                exit_mask,
            )
        ),
        "presence_mae": (
            safe_mae(
                y,
                hard_pred,
                presence_mask,
            )
        ),
        "changed_team_mae": (
            safe_mae(
                y,
                hard_pred,
                changed,
            )
        ),
        "10plus_mae": (
            safe_mae(
                y,
                hard_pred,
                ten_mask,
            )
        ),
        "20plus_mae": (
            safe_mae(
                y,
                hard_pred,
                twenty_mask,
            )
        ),
        "a_roc_auc": (
            roc_auc_score(
                presence,
                prob,
            )
        ),
        "a_macro_f1": (
            class_m[
                "macro_f1"
            ]
        ),
        "a_exit_recall": (
            class_m[
                "exit_recall"
            ]
        ),
        "threshold": (
            threshold
        ),
    }

# Part H. 4개 Variant × 4개 Fold 실행

In [20]:
rows = []

for (
    variant,
    cfg,
) in (
    FEATURE_VARIANTS.items()
):
    print(
        "\n"
        + "=" * 90
    )

    print(
        variant
    )

    for fold, val_season in enumerate(
        OUTER_VAL_SEASONS,
        start=1,
    ):
        (
            train_df,
            val_df,
        ) = split_outer(
            dev,
            val_season,
        )

        (
            prob,
            threshold,
        ) = fit_model_a(
            train_df,
            val_df,
            numeric_features=(
                cfg[
                    "a_numeric"
                ]
            ),
            seed=(
                SEED
                + fold * 100
            ),
        )

        conditional = fit_model_b(
            train_df,
            val_df,
            numeric_features=(
                cfg[
                    "b_numeric"
                ]
            ),
            seed=(
                SEED
                + fold * 1000
            ),
        )

        metrics = evaluate(
            val_df,
            prob,
            threshold,
            conditional,
        )

        rows.append({
            "variant": variant,
            "fold": fold,
            "outer_val_season": (
                val_season
            ),
            **metrics,
        })

        print(
            f"Fold {fold} {val_season} | "
            f"MAE {metrics['mae']:.4f} | "
            f"Exit {metrics['exit_mae']:.4f} | "
            f"Changed {metrics['changed_team_mae']:.4f} | "
            f"10+ {metrics['10plus_mae']:.4f}"
        )


fold_results = pd.DataFrame(
    rows
)


FULL


Default metric period is 5 because MAE is/are not implemented for GPU


Fold 1 2020-2021 | MAE 2.0751 | Exit 0.6419 | Changed 2.1576 | 10+ 5.9309


Default metric period is 5 because MAE is/are not implemented for GPU


Fold 2 2021-2022 | MAE 2.2638 | Exit 1.0764 | Changed 2.6000 | 10+ 6.6487


...

Fold 2 2021-2022 | MAE 2.2795 | Exit 1.1332 | Changed 2.7174 | 10+ 6.7032


Default metric period is 5 because MAE is/are not implemented for GPU


Fold 3 2022-2023 | MAE 2.0667 | Exit 1.0911 | Changed 2.0621 | 10+ 6.4708


Default metric period is 5 because MAE is/are not implemented for GPU


Fold 4 2023-2024 | MAE 2.1267 | Exit 0.4084 | Changed 2.6515 | 10+ 6.9529


# Part I. Summary

In [21]:
summary = (
    fold_results
    .groupby(
        "variant"
    )
    .agg(
        folds=(
            "fold",
            "nunique",
        ),
        mae_mean=(
            "mae",
            "mean",
        ),
        mae_std=(
            "mae",
            "std",
        ),
        mae_worst=(
            "mae",
            "max",
        ),
        rmse_mean=(
            "rmse",
            "mean",
        ),
        r2_mean=(
            "r2",
            "mean",
        ),
        bias_mean=(
            "bias",
            "mean",
        ),
        exit_mae_mean=(
            "exit_mae",
            "mean",
        ),
        presence_mae_mean=(
            "presence_mae",
            "mean",
        ),
        changed_team_mae_mean=(
            "changed_team_mae",
            "mean",
        ),
        tenplus_mae_mean=(
            "10plus_mae",
            "mean",
        ),
        twentyplus_mae_mean=(
            "20plus_mae",
            "mean",
        ),
        a_roc_auc_mean=(
            "a_roc_auc",
            "mean",
        ),
        a_macro_f1_mean=(
            "a_macro_f1",
            "mean",
        ),
        a_exit_recall_mean=(
            "a_exit_recall",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        "mae_mean"
    )
)

summary

,variant,folds,mae_mean,mae_std,mae_worst,rmse_mean,r2_mean,bias_mean,exit_mae_mean,presence_mae_mean,changed_team_mae_mean,tenplus_mae_mean,twentyplus_mae_mean,a_roc_auc_mean,a_macro_f1_mean,a_exit_recall_mean
0,FULL,4,2.126666,0.093655,2.263846,3.258930,0.474459,0.037049,0.810697,2.381066,2.289666,6.462509,10.853575,0.923522,0.833073,0.709783
2,NO_MODEL_A_CONTEXT,4,2.138613,0.097303,2.279456,3.286598,0.465411,0.020825,0.836696,2.391318,2.408811,6.514435,10.961176,0.920510,0.826451,0.714456
1,NO_MARKET,4,2.141627,0.108247,2.302006,3.279310,0.467606,0.037197,0.821938,2.398037,2.310119,6.463636,10.971802,0.913666,0.828061,0.697723
3,NO_TRANSFER_CONTEXT,4,2.211233,0.077319,2.303728,3.341644,0.447583,0.084856,1.242125,2.391072,2.880224,6.461594,10.864267,0.874421,0.773369,0.597765


## FULL 대비 변화

In [22]:
full_summary = (
    summary[
        summary[
            "variant"
        ].eq(
            "FULL"
        )
    ]
    .iloc[0]
)

full_fold = (
    fold_results[
        fold_results[
            "variant"
        ].eq(
            "FULL"
        )
    ][
        [
            "fold",
            "mae",
        ]
    ]
    .rename(
        columns={
            "mae": "full_mae",
        }
    )
)

decision_rows = []

for variant in (
    FEATURE_VARIANTS
):
    if variant == (
        "FULL"
    ):
        continue

    row = (
        summary[
            summary[
                "variant"
            ].eq(
                variant
            )
        ]
        .iloc[0]
    )

    variant_fold = (
        fold_results[
            fold_results[
                "variant"
            ].eq(
                variant
            )
        ][
            [
                "fold",
                "mae",
            ]
        ]
        .rename(
            columns={
                "mae": "variant_mae",
            }
        )
    )

    compare = (
        full_fold
        .merge(
            variant_fold,
            on="fold",
        )
    )

    better_folds = int(
        (
            compare[
                "variant_mae"
            ]
            < compare[
                "full_mae"
            ]
        ).sum()
    )

    improvement = float(
        full_summary[
            "mae_mean"
        ]
        - row[
            "mae_mean"
        ]
    )

    material = (
        improvement
        >= MATERIAL_MAE_DELTA
        and better_folds
        >= 3
    )

    decision_rows.append({
        "variant": variant,
        "full_mae": (
            full_summary[
                "mae_mean"
            ]
        ),
        "variant_mae": (
            row[
                "mae_mean"
            ]
        ),
        "improvement_vs_full": (
            improvement
        ),
        "better_folds_vs_full": (
            better_folds
        ),
        "material_threshold": (
            MATERIAL_MAE_DELTA
        ),
        "material_improvement": (
            material
        ),
        "decision": (
            "REVIEW_SIMPLIFICATION"
            if material
            else "KEEP_FULL"
        ),
    })


decision = pd.DataFrame(
    decision_rows
)

decision

,variant,full_mae,variant_mae,improvement_vs_full,better_folds_vs_full,material_threshold,material_improvement,decision
0,NO_MARKET,2.126666,2.141627,-0.014960,0,0.03,False,KEEP_FULL
1,NO_TRANSFER_CONTEXT,2.126666,2.211233,-0.084566,0,0.03,False,KEEP_FULL
2,NO_MODEL_A_CONTEXT,2.126666,2.138613,-0.011947,0,0.03,False,KEEP_FULL


# Part J. 최종 자동 판정

In [23]:
needs_review = (
    decision[
        "material_improvement"
    ].any()
)

if needs_review:
    print(
        "⚠️ 명확한 ablation 개선이 발견됐습니다."
    )

    print(
        "Final Test 전에 해당 group 제거 여부를 한 번 검토합니다."
    )

else:
    print(
        "✅ FULL feature set 안정성 확인."
    )

    print(
        "✅ Feature / model / threshold protocol FREEZE."
    )

    print(
        "➡️ 다음 단계: 11 Final Test + Slice/Error Analysis"
    )

✅ FULL feature set 안정성 확인.
✅ Feature / model / threshold protocol FREEZE.
➡️ 다음 단계: 11 Final Test + Slice/Error Analysis


# Part K. 저장

In [24]:
OUTPUTS = {
    "fold_results": (
        ARTIFACT_DIR
        / "10_feature_stability_fold_results.csv"
    ),
    "summary": (
        ARTIFACT_DIR
        / "10_feature_stability_summary.csv"
    ),
    "decision": (
        ARTIFACT_DIR
        / "10_feature_stability_decision.csv"
    ),
}


fold_results.to_csv(
    OUTPUTS[
        "fold_results"
    ],
    index=False,
)

summary.to_csv(
    OUTPUTS[
        "summary"
    ],
    index=False,
)

decision.to_csv(
    OUTPUTS[
        "decision"
    ],
    index=False,
)


PROTOCOL = {
    "stage": (
        "10 Selected Feature Stability Check"
    ),
    "purpose": (
        "short pre-final-test stability check; "
        "not feature search"
    ),
    "variants": list(
        FEATURE_VARIANTS.keys()
    ),
    "integration": (
        "Hard Two-stage only"
    ),
    "material_mae_delta": (
        MATERIAL_MAE_DELTA
    ),
    "material_rule": (
        "ablation improves mean MAE >= 0.03 "
        "and improves >= 3 of 4 folds"
    ),
    "embedding_experiment": (
        "NOT INCLUDED; final pipeline is CatBoost-based"
    ),
    "hpo": (
        "NOT INCLUDED"
    ),
    "new_feature_search": (
        "NOT INCLUDED"
    ),
    "test_status": (
        "LOCKED / NOT LOADED"
    ),
    "next_step": (
        "11 Final Test + Slice/Error Analysis"
    ),
}

PROTOCOL_PATH = (
    ARTIFACT_DIR
    / "10_feature_stability_protocol.json"
)

with PROTOCOL_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        PROTOCOL,
        f,
        ensure_ascii=False,
        indent=2,
    )


print(
    "Saved:"
)

for name, path in (
    OUTPUTS.items()
):
    print(
        "-",
        path,
    )

print(
    "-",
    PROTOCOL_PATH,
)

Saved:
- /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/10_feature_stability_fold_results.csv
- /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/10_feature_stability_summary.csv
- /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/10_feature_stability_decision.csv
- /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/10_feature_stability_protocol.json


# 실행 후 보내줄 파일

딱 3개만 보내면 됩니다.

```text
10_feature_stability_summary.csv
10_feature_stability_decision.csv
10_feature_stability_fold_results.csv
```

결과가 `KEEP_FULL`이면 **바로 11 Final Test Notebook으로 진행**합니다.